In [ ]:
import numpy as np

class ILS:
    def __init__(self, objective_function, bounds, perturbation_scale=0.1, max_iters=100, local_search_iters=50, step_size=0.05):
        self.f = objective_function
        self.bounds = np.array(bounds)
        self.dim = len(bounds)
        self.perturbation_scale = perturbation_scale
        self.max_iters = max_iters
        self.ls_iters = local_search_iters
        self.step_size = step_size

    def _hill_climbing(self, current_solution):
        best_sol = np.array(current_solution)
        best_fitness = self.f(best_sol)
        
        for _ in range(self.ls_iters):
            step = np.random.normal(0, self.step_size, self.dim)
            candidate = best_sol + step
            candidate = np.clip(candidate, self.bounds[:, 0], self.bounds[:, 1])
            
            candidate_fitness = self.f(candidate)
            if candidate_fitness < best_fitness:
                best_sol = candidate
                best_fitness = candidate_fitness
                
        return best_sol

    def _perturb(self, solution):
        perturbed = np.array(solution)
        ranges = self.bounds[:, 1] - self.bounds[:, 0]
        
        # Perturbación controlada usando ruido uniforme basado en la escala
        noise = np.random.uniform(-1, 1, self.dim) * (ranges * self.perturbation_scale)
        perturbed += noise
        
        return np.clip(perturbed, self.bounds[:, 0], self.bounds[:, 1])

    def _accept(self, current_sol, candidate_sol):
        if self.f(candidate_sol) < self.f(current_sol):
            return candidate_sol
        return current_sol

    def run(self):
        # 1. Inicialización
        s0 = np.random.uniform(self.bounds[:, 0], self.bounds[:, 1], self.dim)
        s_star = self._hill_climbing(s0)
        
        best_sol = s_star.copy()
        best_fitness = self.f(best_sol)

        for _ in range(self.max_iters):
            # 2. Perturbación
            s_prime = self._perturb(s_star)
            
            # 3. Búsqueda local
            s_prime_star = self._hill_climbing(s_prime)
            
            # 4. Criterio de aceptación
            s_star = self._accept(s_star, s_prime_star)

            current_fitness = self.f(s_star)
            if current_fitness < best_fitness:
                best_sol = s_star.copy()
                best_fitness = current_fitness

        return best_sol, best_fitness